# MLGT Algorithm Investigation

This notebook investigates the behavior of the MLGT algorithm in two locations, fixes nopython errors, and analyzes hash match distributions.

**Outline:**
1. Fix Nopython Errors in Algorithm
2. Import and Compare `mlgt.py` Implementations
3. Test Thresholds for Hash Matching
4. Compare Number of Hash Matches (One Query vs Dataset)
5. Plot Distribution of Hash Matches as Histogram

In [1]:
import os
import sys
import gc
from argparse import ArgumentParser, Namespace

import math
import cmath
import random
import time

from tqdm import tqdm, trange
from typing import List, Tuple, Dict, Set, Union, Optional, Literal, Any, Iterable, Callable, NamedTuple
from dataclasses import dataclass, field

import numpy as np
from numpy import array, ndarray, linalg, matrix, strings, testing

import matplotlib as mpl
from matplotlib import pyplot as plt, ticker

import numba
from numba.typed import List as NumbaList, Dict as NumbaDict
from numba import types, prange

from py_src.sim_hash import SimHash, test_simhash

# Example: Efficient parallelized function with numba.jit
@numba.njit(parallel=True)
def parallel_dot(a, b):
    result = np.zeros(a.shape[0])
    for i in prange(a.shape[0]):
        result[i] = np.dot(a[i], b)
    return result

In [2]:
hash_function = SimHash(
    num_hashes=500,
    num_bits=10,
    threshold=0,
    dimension=1000
)

mtp_dataset: ndarray = np.load(os.path.expanduser("~/MTP/dataset.npy"), mmap_mode='r')
mtp_query_set: ndarray = np.load(os.path.expanduser("~/MTP/dataset_test.npy"), mmap_mode='r')
imagenet_dataset: ndarray = np.load(os.path.expanduser("~/tgtnn/data/imagenet/X.npy"), mmap_mode='r')
imagenet_query_set: ndarray = np.load(os.path.expanduser("~/tgtnn/data/imagenet/Q.npy"), mmap_mode='r')

In [3]:
# Compute hashes
mtp_hashes: ndarray = hash_function(mtp_dataset)
mtp_query_hashes: ndarray = hash_function(mtp_query_set)
imagenet_hashes: ndarray = hash_function(imagenet_dataset)
imagenet_query_hashes: ndarray = hash_function(imagenet_query_set)

In [4]:
print("dataset_hashes shape:", mtp_hashes.shape, "dtype:", mtp_hashes.dtype)
print("query_hashes shape:", mtp_query_hashes.shape, "dtype:", mtp_query_hashes.dtype)
print("dataset_hashes sample:", mtp_hashes[0, :10])
print("query_hashes sample:", mtp_query_hashes[0, :10])
print("Any overlap?", np.any(mtp_hashes == mtp_query_hashes[0]))
print(np.sum(mtp_hashes[0] == mtp_query_hashes[0]))  # Should be >0 if there are matches

dataset_hashes shape: (390000, 500, 10) dtype: bool
query_hashes shape: (3000, 500, 10) dtype: bool
dataset_hashes sample: [[ True  True False False  True False False  True  True False]
 [ True False False False False  True False  True  True False]
 [False False False False  True  True False  True False False]
 [ True False False False  True False  True False  True  True]
 [False False  True False  True  True  True False False  True]
 [False  True False  True False False False False  True  True]
 [ True False  True  True False False False False  True  True]
 [ True False  True False False False False  True False False]
 [ True  True False False False  True  True  True False False]
 [False False  True False False  True False  True  True  True]]
query_hashes sample: [[False False False  True False  True  True  True  True  True]
 [False  True False  True  True  True  True  True  True  True]
 [False False  True  True False  True False False False False]
 [ True  True False False False  Tru

In [6]:
# For MTP dataset: get histogram of number of matches across 100 queries
mtp_matches: ndarray = np.zeros((100, mtp_hashes.shape[0]), dtype=np.int32)
for qidx in tqdm(range(100), desc="Comparing MTP query hashes"):
    mtp_matches[qidx] = hash_function.compare_hashes(mtp_hashes, mtp_query_hashes[qidx]).astype(np.int32).sum(axis=1)

assert mtp_matches.dtype == np.int32
flattened_matches = mtp_matches.flatten()
hist_counts = np.bincount(flattened_matches, minlength=501)
print(hist_counts)

Comparing MTP query hashes: 100%|██████████| 100/100 [13:31<00:00,  8.12s/it]


[22369869 11856842  3346267   748181   195255    84392    50189    34374
    26940    22002    18163    16500    13931    12734    11327     9926
     9080     7648     6915     6350     5845     5290     5156     4777
     4341     3922     3614     3323     3057     3054     2981     2658
     2719     2514     2390     2149     2022     1934     1818     1764
     1718     1690     1668     1652     1969     1814     1779     1762
     1864     1625     1404     1228     1163     1062      952      919
      911      834      816      723      776      776      695      727
      679      637      606      599      636      585      512      563
      568      525      462      547      509      471      493      480
      448      440      446      407      417      398      396      443
      382      386      396      389      369      372      354      357
      367      384      380      335      353      345      277      373
      285      304      289      290      290      

In [ ]:
# For ImageNet dataset: get histogram of number of matches across 100 queries
imagenet_matches: ndarray = np.zeros((100, imagenet_hashes.shape[0]), dtype=np.int32)
for qidx in tqdm(range(100), desc="Comparing ImageNet query hashes"):
    imagenet_matches[qidx] = hash_function.compare_hashes(imagenet_hashes, imagenet_query_hashes[qidx]).astype(np.int32).sum(axis=1)

assert imagenet_matches.dtype == np.int32
flattened_matches = imagenet_matches.flatten()
hist_counts = np.bincount(flattened_matches, minlength=501)
print(hist_counts)

Comparing ImageNet query hashes:  89%|████████▉ | 89/100 [38:02<04:50, 26.41s/it]